### Construção do dataset final

Monta o dataset a partir dos intermediários (histórico). **Para o dataset usado na ML**, rode depois `03_realign_features_by_electrode.ipynb` (features por nome de eletrodo).

Fluxo:
1. Lê features intermediárias (`data/intermediate/sub-*.parquet`)
2. Faz merge com metadados (`participants.csv`)
3. Aplica filtro de amostra saudável
4. Gera dummies de metadados
5. Salva `data/processed/dataset_features.parquet`

Pré-requisito: `01_build_networks.ipynb`. Ordem recomendada: `01` → `03_realign` → EDA/ML (este `02` documenta o merge/filtro; o `03` regenera o parquet final alinhado).


### Imports


In [1]:
from pathlib import Path

import pandas as pd


### Paths


In [2]:
INTERMEDIATE_DIR = Path("../../data/intermediate")
METADATA_FILE = Path("../../data/metadata/participants.csv")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "dataset_features.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR.resolve())
print("METADATA_FILE:", METADATA_FILE.resolve())
print("OUTPUT_FILE:", OUTPUT_FILE.resolve())


INTERMEDIATE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate
METADATA_FILE: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\metadata\participants.csv
OUTPUT_FILE: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features.parquet


### Carregar features intermediárias


In [3]:
print("Lendo features intermediárias...")
files = sorted(INTERMEDIATE_DIR.glob("sub-*.parquet"))
print(f"Encontrados {len(files)} arquivos")

dfs = []
for file in files:
    subject_id, condition = file.stem.split("_", 1)
    df = pd.read_parquet(file)
    df["subject_id"] = subject_id
    df["condition"] = condition
    dfs.append(df)

df_features = pd.concat(dfs, ignore_index=True)
df_features.shape


Lendo features intermediárias...
Encontrados 236 arquivos


(236, 375)

### Merge com metadados


In [4]:
print("Lendo metadados...")
meta = pd.read_csv(METADATA_FILE)
meta["subject_id"] = meta["ID"]

df_final = df_features.merge(
    meta,
    on="subject_id",
    how="left",
    validate="m:1",
)

Lendo metadados...


In [5]:
df_final

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
0,52663.0,50136.0,64259.0,51050.0,47185.0,50810.0,66577.0,68306.0,61874.0,64524.0,...,"#CODE:296.26 #DESC:Major Depressive Disorder,...",NaN,past subclinical dperession,0.0,7.0,1.0,7.0,29.5,No,No
1,62390.0,61541.0,65256.0,60471.0,61281.0,62008.0,64234.0,65951.0,56157.0,58993.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,6.0,2.0,11.0,53.5,Yes (9),No
2,60848.0,54287.0,63370.0,55184.0,55336.0,53932.0,59144.0,63075.0,47081.0,51530.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,6.0,2.0,11.0,53.5,Yes (9),No
3,69181.0,65135.0,69459.0,61255.0,59635.0,62813.0,70100.0,66427.0,63006.0,71578.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,3.0,1.0,3.0,5,No,Yes
4,64208.0,64015.0,62572.0,58776.0,54827.0,58702.0,70764.0,65421.0,62396.0,72140.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,3.0,1.0,3.0,5,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,47030.0,45225.0,44604.0,48978.0,44789.0,40293.0,30946.0,48379.0,44559.0,40355.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,0.0,0.0,0.0,3.0,6,Yes,Yes
232,56287.0,42899.0,60575.0,57076.0,45056.0,54058.0,50954.0,65584.0,52414.0,46365.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,1.0,0.0,3.0,26,No,No
233,62478.0,52541.0,61065.0,52796.0,37890.0,43432.0,58213.0,68323.0,46152.0,41945.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,4.0,1.0,0.0,3.0,26,No,No
234,65972.0,64072.0,75547.0,59057.0,43914.0,57899.0,71752.0,73206.0,43058.0,55037.0,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,NaN,NaN,2.0,8.0,0.0,2.0,12,yes,yes


In [6]:

df_final = df_final.rename(
    columns={
        "Gender_ 1=female_2=male": "gender",
        "Age": "age",
        "Handedness": "handedness",
        "Education": "education",
        "DRUG_0=negative_1=Positive": "drug",
        "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
        "SKID_Diagnoses": "skid_diagnoses",
        "SKID_Diagnoses 1": "skid_diagnoses_1",
        "SKID_Diagnoses 2": "skid_diagnoses_2",
        "Comments_SKID_assessment": "comments_skid_assessment",
        "Hamilton_Scale": "hamilton_scale",
        "BSL23_sumscore": "bsl23_sumscore",
        "BSL23_behavior": "bsl23_behavior",
        "AUDIT": "audit",
        "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
        "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
        "Relationship_Status": "relationship_status",
    }
)

print(f"Shape inicial: {df_final.shape}")
df_final[["subject_id", "condition", "age", "gender"]].head()


Shape inicial: (236, 396)


,subject_id,condition,age,gender
0,sub-010060,EO,30-35,2
1,sub-010061,EC,25-30,2
2,sub-010061,EO,25-30,2
3,sub-010062,EC,25-30,1
4,sub-010062,EO,25-30,1


### Padronizar strings e filtrar amostra saudável


In [7]:
df_final["relationship_status"] = df_final["relationship_status"].str.lower()
df_final["handedness"] = df_final["handedness"].str.lower()
df_final["education"] = df_final["education"].str.lower()

healthy_mask = (
    (df_final["drug"] == 0)
    & (df_final["smoking"] == 1)
    & (df_final["audit"] < 8)
    & (df_final["hamilton_scale"] <= 7)
    & (df_final["bsl23_behavior"] == 0)
    & (df_final["skid_diagnoses"].str.lower() == "none")
    & (df_final["skid_diagnoses_2"].isna())
    & (df_final["comments_skid_assessment"].isna())
)

df_healthy = df_final.loc[healthy_mask].copy()

print(f"Shape após filtro saudável: {df_healthy.shape}")
print(f"N indivíduos antes do filtro: {df_final['subject_id'].nunique()}")
print(f"N indivíduos após filtro saudável: {df_healthy['subject_id'].nunique()}")


Shape após filtro saudável: (108, 396)
N indivíduos antes do filtro: 119
N indivíduos após filtro saudável: 54


### Dummies de metadados


In [8]:
META_COLS = [
    "age",
    "gender",
    "handedness",
    "education",
    "relationship_status",
]

df_meta = df_healthy[META_COLS].copy()
print(f"Shape após seleção de metadados: {df_meta.shape}")

df_meta = pd.get_dummies(df_meta, columns=META_COLS, drop_first=True)
df_meta.head()


Shape após seleção de metadados: (108, 5)


,age_25-30,age_30-35,age_55-60,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
17,False,False,False,False,False,False,False,True,False,True,True,False,False
18,False,False,False,False,False,False,False,True,False,True,True,False,False
19,False,False,False,False,False,False,False,True,False,True,False,True,True
20,False,False,False,False,False,False,False,True,False,True,False,True,True
33,True,False,False,False,False,False,False,False,True,False,True,False,False


### Concatenar, corrigir E0 → EO e salvar


In [ ]:
df_analysis = pd.concat([df_healthy, df_meta], axis=1)
df_analysis["condition"] = df_analysis["condition"].replace({"E0": "EO"})

print(f"Shape após concatenação: {df_analysis.shape}")
print(df_analysis["condition"].value_counts())

df_analysis.to_parquet(OUTPUT_FILE, index=False)
print(f"Dataset final salvo em: {OUTPUT_FILE.resolve()}")
print(f"Shape final: {df_analysis.shape}")
df_analysis.head()


Shape após concatenação: (108, 409)
condition
EC    54
EO    54
Name: count, dtype: int64
Dataset final salvo em: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features.parquet
Shape final: (108, 409)


,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
17,43398.0,43122.0,39397.0,37806.0,34149.0,42944.0,40750.0,29859.0,35429.0,42523.0,...,False,False,False,False,True,False,True,True,False,False
18,50171.0,50484.0,55432.0,46863.0,40978.0,42528.0,48873.0,43130.0,40329.0,61632.0,...,False,False,False,False,True,False,True,True,False,False
19,68169.0,69894.0,68886.0,63585.0,56792.0,63773.0,72372.0,68516.0,53248.0,61239.0,...,False,False,False,False,True,False,True,False,True,True
20,46291.0,47086.0,58607.0,50972.0,47997.0,54619.0,61026.0,63287.0,50952.0,57500.0,...,False,False,False,False,True,False,True,False,True,True
33,68411.0,70127.0,71233.0,66132.0,66503.0,65958.0,67654.0,62060.0,55974.0,52385.0,...,False,False,False,False,False,True,False,True,False,False


: 

### Próximo passo

Com o dataset pronto, use os notebooks de EDA:

- `notebook/01_eda/01_eda_global.ipynb`
- `notebook/01_eda/02_eda_node_features.ipynb`
- `notebook/01_eda/03_analise_descriva_redes.ipynb`
